In [1]:
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 1.5 MB/s eta 0:00:0000:0100:01


In [4]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password="Password@2109",
    database="retail_analysis"
)

cursor = conn.cursor()

In [5]:
cursor.execute("SELECT VERSION();")
print(cursor.fetchone())

('9.6.0',)


In [6]:
cursor.execute("""
SELECT COUNT(*) AS Total_Rows
FROM sales;
""")

print(cursor.fetchone())

(0,)


In [7]:
cursor.execute("SELECT DATABASE();")
print(cursor.fetchone())

('retail_analysis',)


In [8]:
cursor.execute("SHOW TABLES;")
print(cursor.fetchall())

[('sales',)]


In [10]:
import pandas as pd

df_sales = pd.read_csv(
    "/Users/shivi/online_retail_sql.csv",
    encoding="utf-8"
)

print(df_sales.shape)

(504731, 16)


/var/folders/82/6rq2dldj5vxfz2191z6z1jfr0000gn/T/ipykernel_89005/2919801650.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sales = pd.read_csv(


In [11]:
print(df_sales.columns.tolist())

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer_ID', 'Country', 'Transaction_Type', 'Revenue', 'Year', 'Month', 'Month_Name', 'Year_Month', 'Day_Of_Week', 'Hour']


In [12]:
df_sales.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,Transaction_Type,Revenue,Year,Month,Month_Name,Year_Month,Day_Of_Week,Hour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Sale,83.4,2009,12,December,2009-12,Tuesday,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Sale,81.0,2009,12,December,2009-12,Tuesday,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Sale,81.0,2009,12,December,2009-12,Tuesday,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Sale,100.8,2009,12,December,2009-12,Tuesday,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Sale,30.0,2009,12,December,2009-12,Tuesday,7


In [13]:
print(df_sales.shape)
print(df_sales.isnull().sum())

(504731, 16)
Invoice                  0
StockCode                0
Description              0
Quantity                 0
InvoiceDate              0
Price                    0
Customer_ID         103815
Country                  0
Transaction_Type         0
Revenue                  0
Year                     0
Month                    0
Month_Name               0
Year_Month               0
Day_Of_Week              0
Hour                     0
dtype: int64


In [14]:
df_sql = df_sales.copy()

df_sql["Customer_ID"] = df_sql["Customer_ID"].where(
    df_sql["Customer_ID"].notna(),
    None
)

df_sql["InvoiceDate"] = pd.to_datetime(df_sql["InvoiceDate"])

print(df_sql.shape)

(504731, 16)


In [15]:
cursor.execute("SELECT DATABASE();")
print(cursor.fetchone())

('retail_analysis',)


In [16]:
cursor.execute("TRUNCATE TABLE sales")
conn.commit()

cursor.execute("SELECT COUNT(*) FROM sales")
print(cursor.fetchone())

(0,)


In [17]:
insert_query = """
INSERT INTO sales (
    Invoice,
    StockCode,
    Description,
    Quantity,
    InvoiceDate,
    Price,
    Customer_ID,
    Country,
    Transaction_Type,
    Revenue,
    Sales_Year,
    Sales_Month,
    Month_Name,
    Sales_Year_Month,
    Day_Of_Week,
    Sales_Hour
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

In [18]:
data = list(
    df_sql[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer_ID",
            "Country",
            "Transaction_Type",
            "Revenue",
            "Year",
            "Month",
            "Month_Name",
            "Year_Month",
            "Day_Of_Week",
            "Hour"
        ]
    ].itertuples(index=False, name=None)
)

In [19]:
batch_size = 5000

for i in range(0, len(data), batch_size):
    batch = data[i:i + batch_size]

    cursor.executemany(insert_query, batch)
    conn.commit()

    print(f"Inserted {min(i + batch_size, len(data))} / {len(data)}")

ProgrammingError: 1054 (42S22): Unknown column 'nan' in 'field list'

In [20]:
import pandas as pd
import numpy as np

df_sql = df_sales.copy()

# Convert every NaN/NaT to Python None
df_sql = df_sql.replace({np.nan: None})

In [21]:
df_sql["InvoiceDate"] = pd.to_datetime(
    df_sql["InvoiceDate"],
    errors="coerce"
)

df_sql["InvoiceDate"] = df_sql["InvoiceDate"].apply(
    lambda x: x.to_pydatetime() if pd.notna(x) else None
)

In [22]:
print(df_sql.isna().sum())

Invoice                  0
StockCode                0
Description              0
Quantity                 0
InvoiceDate              0
Price                    0
Customer_ID         103815
Country                  0
Transaction_Type         0
Revenue                  0
Year                     0
Month                    0
Month_Name               0
Year_Month               0
Day_Of_Week              0
Hour                     0
dtype: int64


In [23]:
import pandas as pd
import numpy as np

df_sql = df_sales.copy()

# Convert ALL missing values to Python None
df_sql = df_sql.astype(object).where(pd.notna(df_sql), None)

print(df_sql.isna().sum())

Invoice                  0
StockCode                0
Description              0
Quantity                 0
InvoiceDate              0
Price                    0
Customer_ID         103815
Country                  0
Transaction_Type         0
Revenue                  0
Year                     0
Month                    0
Month_Name               0
Year_Month               0
Day_Of_Week              0
Hour                     0
dtype: int64


In [24]:
columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer_ID",
    "Country",
    "Transaction_Type",
    "Revenue",
    "Year",
    "Month",
    "Month_Name",
    "Year_Month",
    "Day_Of_Week",
    "Hour"
]

data = [
    tuple(row)
    for row in df_sql[columns].itertuples(index=False, name=None)
]

print("Rows ready:", len(data))

Rows ready: 504731


In [25]:
cursor.execute("TRUNCATE TABLE sales")
conn.commit()

In [26]:
insert_query = """
INSERT INTO sales (
    Invoice,
    StockCode,
    Description,
    Quantity,
    InvoiceDate,
    Price,
    Customer_ID,
    Country,
    Transaction_Type,
    Revenue,
    Sales_Year,
    Sales_Month,
    Month_Name,
    Year_Month,
    Day_Of_Week,
    Sales_Hour
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s, %s, %s, %s)
"""

batch_size = 5000

for i in range(0, len(data), batch_size):
    batch = data[i:i + batch_size]

    cursor.executemany(insert_query, batch)
    conn.commit()

    print(f"{min(i + batch_size, len(data))} / {len(data)}")

ProgrammingError: 1064 (42000): You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near 'Year_Month,
    Day_Of_Week,
    Sales_Hour
)
VALUES (489434, '85048', '15CM CHR' at line 15

In [27]:
cursor.execute("DESCRIBE sales")

for row in cursor.fetchall():
    print(row)

('Invoice', 'varchar(20)', 'YES', '', None, '')
('StockCode', 'varchar(20)', 'YES', '', None, '')
('Description', 'varchar(255)', 'YES', '', None, '')
('Quantity', 'int', 'YES', '', None, '')
('InvoiceDate', 'datetime', 'YES', '', None, '')
('Price', 'decimal(12,3)', 'YES', '', None, '')
('Customer_ID', 'varchar(20)', 'YES', '', None, '')
('Country', 'varchar(100)', 'YES', '', None, '')
('Transaction_Type', 'varchar(20)', 'YES', '', None, '')
('Revenue', 'decimal(15,3)', 'YES', '', None, '')
('Sales_Year', 'int', 'YES', '', None, '')
('Sales_Month', 'int', 'YES', '', None, '')
('Month_Name', 'varchar(20)', 'YES', '', None, '')
('Sales_Year_Month', 'varchar(10)', 'YES', '', None, '')
('Day_Of_Week', 'varchar(20)', 'YES', '', None, '')
('Sales_Hour', 'int', 'YES', '', None, '')


In [28]:
cursor.execute("DROP TABLE IF EXISTS sales")
conn.commit()

In [29]:
create_table = """
CREATE TABLE sales (
    Invoice VARCHAR(20),
    StockCode VARCHAR(20),
    Description VARCHAR(255),
    Quantity INT,
    InvoiceDate DATETIME,
    Price DECIMAL(12,3),
    Customer_ID INT NULL,
    Country VARCHAR(100),
    Transaction_Type VARCHAR(30),
    Revenue DECIMAL(15,3),
    Sales_Year INT,
    Sales_Month INT,
    Month_Name VARCHAR(20),
    Sales_Year_Month VARCHAR(10),
    Day_Of_Week VARCHAR(20),
    Sales_Hour INT
)
"""

cursor.execute(create_table)
conn.commit()

print("Sales table created successfully")

Sales table created successfully


In [30]:
insert_query = """
INSERT INTO sales (
    `Invoice`,
    `StockCode`,
    `Description`,
    `Quantity`,
    `InvoiceDate`,
    `Price`,
    `Customer_ID`,
    `Country`,
    `Transaction_Type`,
    `Revenue`,
    `Sales_Year`,
    `Sales_Month`,
    `Month_Name`,
    `Sales_Year_Month`,
    `Day_Of_Week`,
    `Sales_Hour`
)
VALUES (
    %s, %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s, %s, %s
)
"""

In [31]:
df_sql = df_sales.copy()

# Convert NaN / NaT to None
df_sql = df_sql.astype(object).where(pd.notna(df_sql), None)

# Convert InvoiceDate to Python datetime
df_sql["InvoiceDate"] = pd.to_datetime(
    df_sql["InvoiceDate"],
    errors="coerce"
)

df_sql["InvoiceDate"] = df_sql["InvoiceDate"].apply(
    lambda x: x.to_pydatetime() if pd.notna(x) else None
)

In [32]:
columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer_ID",
    "Country",
    "Transaction_Type",
    "Revenue",
    "Year",
    "Month",
    "Month_Name",
    "Year_Month",
    "Day_Of_Week",
    "Hour"
]

data = []

for row in df_sql[columns].itertuples(index=False, name=None):
    clean_row = tuple(
        None if pd.isna(value) else value
        for value in row
    )
    data.append(clean_row)

print("Rows ready:", len(data))

Rows ready: 504731


In [33]:
batch_size = 5000

for i in range(0, len(data), batch_size):

    batch = data[i:i + batch_size]

    cursor.executemany(insert_query, batch)
    conn.commit()

    print(f"Inserted {min(i + batch_size, len(data))} / {len(data)}")

Inserted 5000 / 504731
Inserted 10000 / 504731
Inserted 15000 / 504731
Inserted 20000 / 504731
Inserted 25000 / 504731
Inserted 30000 / 504731
Inserted 35000 / 504731
Inserted 40000 / 504731
Inserted 45000 / 504731
Inserted 50000 / 504731
Inserted 55000 / 504731
Inserted 60000 / 504731
Inserted 65000 / 504731
Inserted 70000 / 504731
Inserted 75000 / 504731
Inserted 80000 / 504731
Inserted 85000 / 504731
Inserted 90000 / 504731
Inserted 95000 / 504731
Inserted 100000 / 504731
Inserted 105000 / 504731
Inserted 110000 / 504731
Inserted 115000 / 504731
Inserted 120000 / 504731
Inserted 125000 / 504731
Inserted 130000 / 504731
Inserted 135000 / 504731
Inserted 140000 / 504731
Inserted 145000 / 504731
Inserted 150000 / 504731
Inserted 155000 / 504731
Inserted 160000 / 504731
Inserted 165000 / 504731
Inserted 170000 / 504731
Inserted 175000 / 504731
Inserted 180000 / 504731
Inserted 185000 / 504731
Inserted 190000 / 504731
Inserted 195000 / 504731
Inserted 200000 / 504731
Inserted 205000 / 50

In [34]:
cursor.execute("SELECT COUNT(*) FROM sales")

print(cursor.fetchone())

(504731,)


In [35]:
cursor.execute("""
SELECT
    COUNT(*) AS Total_Rows,
    COUNT(DISTINCT Invoice) AS Transactions,
    COUNT(DISTINCT Customer_ID) AS Customers,
    COUNT(DISTINCT StockCode) AS Products
FROM sales
""")

print(cursor.fetchone())

(504731, 20952, 4312, 4102)


# Phase 1 — Verify the SQL database

In [37]:
cursor.execute("""
SELECT
    COUNT(*) AS Total_Rows,
    COUNT(DISTINCT Invoice) AS Transactions,
    COUNT(DISTINCT Customer_ID) AS Customers,
    COUNT(DISTINCT StockCode) AS Products,
    COUNT(DISTINCT Country) AS Countries,
    ROUND(SUM(Revenue), 2) AS Total_Revenue
FROM sales;
""")

result = cursor.fetchone()

print(result)

(504731, 20952, 4312, 4102, 40, Decimal('10272136.23'))


In [38]:
cursor.execute("SELECT COUNT(*) FROM sales;")
print(cursor.fetchone())

(504731,)


# Start Business Question 1 : Which countries generate the most revenue?

In [41]:
cursor.execute("""
SELECT
    Country,
    ROUND(SUM(Revenue), 2) AS Total_Revenue,
    COUNT(DISTINCT Invoice) AS Transactions,
    COUNT(DISTINCT Customer_ID) AS Customers,
    ROUND(SUM(Revenue)/ COUNT(DISTINCT Invoice),2) AS AOV
    
FROM sales
GROUP BY Country
ORDER BY Total_Revenue DESC;
""")

results = cursor.fetchall()

for row in results[:10]:
    print(row)

('United Kingdom', Decimal('8812685.40'), 19291, 3969, Decimal('456.83'))
('EIRE', Decimal('380909.57'), 348, 5, Decimal('1094.57'))
('Netherlands', Decimal('268784.35'), 135, 22, Decimal('1991.00'))
('Germany', Decimal('202025.39'), 347, 67, Decimal('582.21'))
('France', Decimal('147103.14'), 241, 47, Decimal('610.39'))
('Sweden', Decimal('53501.99'), 69, 16, Decimal('775.39'))
('Denmark', Decimal('50906.85'), 26, 9, Decimal('1957.96'))
('Spain', Decimal('47568.65'), 66, 25, Decimal('720.74'))
('Switzerland', Decimal('43921.39'), 40, 14, Decimal('1098.03'))
('Australia', Decimal('31446.80'), 40, 15, Decimal('786.17'))


# Business Question 2 : Which products generate the highest revenue?

In [42]:
cursor.execute("""
SELECT
    StockCode,
    Description,
    ROUND(SUM(Revenue), 2) AS Revenue,
    SUM(Quantity) AS Units_Sold,
    COUNT(DISTINCT Invoice) AS Transactions
FROM sales
GROUP BY StockCode, Description
ORDER BY Revenue DESC
LIMIT 10;
""")

results = cursor.fetchall()

for row in results:
    print(row)

('M', 'Manual', Decimal('262963.43'), Decimal('2769'), 516)
('22423', 'REGENCY CAKESTAND 3 TIER', Decimal('169912.76'), Decimal('13685'), 2019)
('85123A', 'WHITE HANGING HEART T-LIGHT HOLDER', Decimal('160345.63'), Decimal('58691'), 3315)
('DOT', 'DOTCOM POSTAGE', Decimal('116408.71'), Decimal('730'), 730)
('84879', 'ASSORTED COLOUR BIRD ORNAMENT', Decimal('72890.19'), Decimal('45228'), 1412)
('22086', "PAPER CHAIN KIT 50'S CHRISTMAS ", Decimal('58127.30'), Decimal('17205'), 957)
('85099B', 'JUMBO BAG RED RETROSPOT', Decimal('56480.46'), Decimal('30746'), 1246)
('47566', 'PARTY BUNTING', Decimal('49664.12'), Decimal('10079'), 1013)
('POST', 'POSTAGE', Decimal('49477.54'), Decimal('2310'), 759)
('84347', 'ROTATING SILVER ANGELS T-LIGHT HLDR', Decimal('47954.49'), Decimal('23037'), 347)


# Business Question 3 : Top 10 Customers by Revenue

In [43]:
cursor.execute("""
SELECT
    Customer_ID,
    ROUND(SUM(Revenue), 2) AS Revenue,
    COUNT(DISTINCT Invoice) AS Transactions,
    SUM(Quantity) AS Units_Sold
FROM sales
WHERE Customer_ID IS NOT NULL
GROUP BY Customer_ID
ORDER BY Revenue DESC
LIMIT 10;
""")

results = cursor.fetchall()

for row in results:
    print(row)

(18102, Decimal('349164.35'), 89, Decimal('124216'))
(14646, Decimal('248396.50'), 78, Decimal('170278'))
(14156, Decimal('196549.74'), 102, Decimal('108105'))
(14911, Decimal('152121.22'), 205, Decimal('69709'))
(13694, Decimal('131443.19'), 94, Decimal('125893'))
(17511, Decimal('84541.17'), 31, Decimal('55107'))
(15061, Decimal('83284.38'), 86, Decimal('51791'))
(16684, Decimal('80489.21'), 27, Decimal('54555'))
(16754, Decimal('65500.07'), 29, Decimal('63551'))
(17949, Decimal('60117.60'), 74, Decimal('30112'))


# Business Question 4 : Repeat vs One-Time Customers

In [44]:
cursor.execute("""
WITH customer_orders AS (
    SELECT
        Customer_ID,
        COUNT(DISTINCT Invoice) AS Transactions,
        SUM(Revenue) AS Revenue
    FROM sales
    WHERE Customer_ID IS NOT NULL
    GROUP BY Customer_ID
)
SELECT
    CASE
        WHEN Transactions = 1 THEN 'One-Time Customer'
        ELSE 'Repeat Customer'
    END AS Customer_Type,
    COUNT(*) AS Customers,
    ROUND(SUM(Revenue), 2) AS Revenue,
    ROUND(
        SUM(Revenue) * 100 /
        (SELECT SUM(Revenue) FROM customer_orders),
        2
    ) AS Revenue_Percentage
FROM customer_orders
GROUP BY Customer_Type;
""")

results = cursor.fetchall()

for row in results:
    print(row)

('Repeat Customer', 2893, Decimal('8301929.27'), Decimal('94.36'))
('One-Time Customer', 1419, Decimal('496304.47'), Decimal('5.64'))


# Business Question 5: How does revenue change over time, and which months are strongest?

In [45]:
cursor.execute("""
SELECT
    Sales_Year,
    Sales_Month,
    Sales_Year_Month,
    ROUND(SUM(Revenue), 2) AS Revenue,
    COUNT(DISTINCT Invoice) AS Transactions
FROM sales
GROUP BY
    Sales_Year,
    Sales_Month,
    Sales_Year_Month
ORDER BY
    Sales_Year,
    Sales_Month;
""")

results = cursor.fetchall()

for row in results:
    print(row)

(2009, 12, '2009-12', Decimal('822483.95'), 1682)
(2010, 1, '2010-01', Decimal('651155.11'), 1105)
(2010, 2, '2010-02', Decimal('551878.30'), 1202)
(2010, 3, '2010-03', Decimal('830915.26'), 1681)
(2010, 4, '2010-04', Decimal('678875.25'), 1462)
(2010, 5, '2010-05', Decimal('657705.50'), 1500)
(2010, 6, '2010-06', Decimal('749537.31'), 1645)
(2010, 7, '2010-07', Decimal('648810.27'), 1529)
(2010, 8, '2010-08', Decimal('695251.91'), 1425)
(2010, 9, '2010-09', Decimal('921696.99'), 1839)
(2010, 10, '2010-10', Decimal('1161902.22'), 2301)
(2010, 11, '2010-11', Decimal('1464293.14'), 2747)
(2010, 12, '2010-12', Decimal('437631.02'), 834)


# Business Question 6: Which days generate the highest revenue and number of transactions?

In [46]:
cursor.execute("""
SELECT
    Day_Of_Week,
    ROUND(SUM(Revenue), 2) AS Revenue,
    COUNT(DISTINCT Invoice) AS Transactions,
    ROUND(
        SUM(Revenue) / COUNT(DISTINCT Invoice),
        2
    ) AS AOV
FROM sales
GROUP BY Day_Of_Week
ORDER BY Revenue DESC;
""")

results = cursor.fetchall()

for row in results:
    print(row)

('Thursday', Decimal('2097097.80'), 4291, Decimal('488.72'))
('Tuesday', Decimal('2002003.30'), 3821, Decimal('523.95'))
('Monday', Decimal('1860809.83'), 3328, Decimal('559.14'))
('Wednesday', Decimal('1745382.10'), 3738, Decimal('466.93'))
('Friday', Decimal('1533319.02'), 3025, Decimal('506.88'))
('Sunday', Decimal('1023721.13'), 2719, Decimal('376.51'))
('Saturday', Decimal('9803.05'), 30, Decimal('326.77'))


# Business Question 7 : At what time of day are customers most active?

In [47]:
cursor.execute("""
SELECT
    Sales_Hour,
    COUNT(DISTINCT Invoice) AS Transactions,
    ROUND(SUM(Revenue), 2) AS Revenue
FROM sales
GROUP BY Sales_Hour
ORDER BY Transactions DESC;
""")

results = cursor.fetchall()

for row in results:
    print(row)

(12, 3341, Decimal('1459647.69'))
(13, 3053, Decimal('1330157.83'))
(14, 2704, Decimal('1199013.91'))
(11, 2594, Decimal('1339629.93'))
(15, 2354, Decimal('1121576.88'))
(10, 2347, Decimal('1180711.35'))
(16, 1449, Decimal('884153.70'))
(9, 1327, Decimal('811807.66'))
(17, 766, Decimal('416238.59'))
(8, 453, Decimal('248940.43'))
(18, 287, Decimal('136836.10'))
(19, 191, Decimal('77876.44'))
(7, 50, Decimal('45173.36'))
(20, 36, Decimal('20372.35'))


# Business Question 8 : Which markets generate the highest-value orders?

In [48]:
cursor.execute("""
SELECT
    Country,
    COUNT(DISTINCT Invoice) AS Transactions,
    COUNT(DISTINCT Customer_ID) AS Customers,
    ROUND(SUM(Revenue), 2) AS Revenue,
    ROUND(
        SUM(Revenue) / COUNT(DISTINCT Invoice),
        2
    ) AS AOV
FROM sales
WHERE Customer_ID IS NOT NULL
GROUP BY Country
HAVING COUNT(DISTINCT Invoice) >= 10
ORDER BY AOV DESC;
""")

results = cursor.fetchall()

for row in results:
    print(row)

('Norway', 11, 5, Decimal('23944.18'), Decimal('2176.74'))
('Netherlands', 135, 22, Decimal('268784.35'), Decimal('1991.00'))
('Denmark', 26, 9, Decimal('50906.85'), Decimal('1957.96'))
('EIRE', 316, 5, Decimal('356041.86'), Decimal('1126.71'))
('Greece', 13, 4, Decimal('14335.67'), Decimal('1102.74'))
('Switzerland', 40, 14, Decimal('43921.39'), Decimal('1098.03'))
('Channel Islands', 30, 11, Decimal('24546.32'), Decimal('818.21'))
('Australia', 40, 15, Decimal('31446.80'), Decimal('786.17'))
('Sweden', 68, 16, Decimal('53147.99'), Decimal('781.59'))
('Spain', 66, 25, Decimal('47568.65'), Decimal('720.74'))
('France', 236, 47, Decimal('146107.07'), Decimal('619.10'))
('Japan', 16, 6, Decimal('9722.02'), Decimal('607.63'))
('Portugal', 40, 18, Decimal('23843.46'), Decimal('596.09'))
('Germany', 347, 67, Decimal('202025.39'), Decimal('582.21'))
('Cyprus', 21, 7, Decimal('11347.10'), Decimal('540.34'))
('Italy', 28, 11, Decimal('15052.73'), Decimal('537.60'))
('Austria', 28, 10, Decimal(

# Business Question 9 : Which products sell in large quantities but generate relatively little revenue per unit?

In [49]:
cursor.execute("""
SELECT
    StockCode,
    Description,
    SUM(Quantity) AS Units_Sold,
    ROUND(SUM(Revenue), 2) AS Revenue,
    ROUND(
        SUM(Revenue) / NULLIF(SUM(Quantity), 0),
        2
    ) AS Revenue_Per_Unit
FROM sales
GROUP BY StockCode, Description
HAVING SUM(Quantity) > 1000
ORDER BY Revenue_Per_Unit ASC
LIMIT 10;
""")

results = cursor.fetchall()

for row in results:
    print(row)

('16053', 'POPART COL BALLPOINT PEN ASST', Decimal('4313'), Decimal('185.73'), Decimal('0.04'))
('35015', 'JACOBS LADDER SMALL', Decimal('2072'), Decimal('134.08'), Decimal('0.06'))
('16044', 'POP-ART FLUORESCENT PENS', Decimal('6193'), Decimal('389.22'), Decimal('0.06'))
('10123G', 'ARMY CAMO WRAPPING TAPE', Decimal('2251'), Decimal('165.56'), Decimal('0.07'))
('16162L', 'THE KING GIFT BAG', Decimal('8081'), Decimal('557.43'), Decimal('0.07'))
('16043', 'POP ART PUSH DOWN RUBBER ', Decimal('2047'), Decimal('168.54'), Decimal('0.08'))
('17038', 'PORCELAIN BUDAH INCENSE HOLDER', Decimal('1134'), Decimal('103.45'), Decimal('0.09'))
('21088', 'SET/6 FRUIT SALAD PAPER CUPS', Decimal('14937'), Decimal('1388.45'), Decimal('0.09'))
('16047', 'POP ART PEN CASE & PENS', Decimal('10612'), Decimal('1054.70'), Decimal('0.10'))
('85110', 'BLACK SILVER FLOWER T-LIGHT HOLDER', Decimal('11520'), Decimal('1138.04'), Decimal('0.10'))


# Business Question 10. Which loyal customers have frequent transactions but relatively low spending per order?

In [50]:
cursor.execute("""
SELECT
    Customer_ID,
    COUNT(DISTINCT Invoice) AS Transactions,
    ROUND(SUM(Revenue), 2) AS Revenue,
    ROUND(
        SUM(Revenue) / COUNT(DISTINCT Invoice),
        2
    ) AS AOV
FROM sales
WHERE Customer_ID IS NOT NULL
GROUP BY Customer_ID
HAVING COUNT(DISTINCT Invoice) >= 10
ORDER BY AOV ASC
LIMIT 10;
""")

results = cursor.fetchall()

for row in results:
    print(row)

(17961, 62, Decimal('1697.85'), Decimal('27.38'))
(12346, 11, Decimal('372.86'), Decimal('33.90'))
(17888, 17, Decimal('1091.44'), Decimal('64.20'))
(16928, 13, Decimal('1199.08'), Decimal('92.24'))
(15611, 13, Decimal('1405.83'), Decimal('108.14'))
(16883, 18, Decimal('2101.61'), Decimal('116.76'))
(17894, 11, Decimal('1292.39'), Decimal('117.49'))
(17969, 18, Decimal('2126.09'), Decimal('118.12'))
(15827, 16, Decimal('1957.98'), Decimal('122.37'))
(15751, 11, Decimal('1380.44'), Decimal('125.49'))


In [52]:
df_sales.to_csv("online_retail_final.csv", index=False)

In [53]:
import os

print(os.getcwd())

/Users/shivi
